# Financial Risk Classification

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

## 1. Load Labeled Data

In [2]:
LABELED_DATA_PATH = 'archive/financial_risk_labeled.csv'
LABEL_COLUMN = 'risk_profile_label'

FEATURES_TO_KEEP = [
    'loan_to_income_ratio', 'expenses_to_income_ratio', 'savings_to_income_ratio',
    'debt_to_income_ratio', 'credit_score', 'previous_default_count',
    'loan_duration_months', 'interest_rate', 'age', 'employment_stability_years'
]


def load_labeled_data(csv_path, seed=SEED):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f'Labeled data not found: {csv_path}. Run clustering.ipynb first.'
        )

    df = pd.read_csv(csv_path)
    required_columns = FEATURES_TO_KEEP + [LABEL_COLUMN]
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f'Missing required columns: {missing_columns}')

    X = df[FEATURES_TO_KEEP].values
    y = df[LABEL_COLUMN].values.astype(int)
    num_classes = len(np.unique(y))

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train)).shuffle(
        10000,
        seed=seed,
        reshuffle_each_iteration=True
    ).batch(32)
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test_scaled, y_test)).batch(32)

    return train_dataset, test_dataset, scaler, FEATURES_TO_KEEP, num_classes

## 2. Custom Component (Custom Layer)

In [3]:
class CustomDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super(CustomDenseLayer, self).__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='random_normal',
            trainable=True,
            name='custom_weight'
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='custom_bias'
        )

    def call(self, inputs):
        output = tf.matmul(inputs, self.w) + self.b
        if self.activation is not None:
            output = self.activation(output)
        return output

## 3. Build Model (Functional API)

In [4]:
def create_model(input_dim, num_classes):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(64, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = CustomDenseLayer(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(16, activation='relu')(x)

    if num_classes == 2:
        outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    else:
        outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='RiskProfile_Model')
    return model

## 4. Custom Training Loop with Gradient Tape

In [5]:
def train_model(
    model,
    train_dataset,
    val_dataset,
    num_classes,
    epochs=60,
    initial_lr=0.001,
    fine_tune_lr=0.0001,
    fine_tune_at=20,
    patience=25,
    min_delta=0.0001
):
    optimizer = tf.keras.optimizers.Adam(learning_rate=initial_lr)

    is_binary = (num_classes == 2)
    if is_binary:
        loss_fn = tf.keras.losses.BinaryCrossentropy()
        acc_metric_train = tf.keras.metrics.BinaryAccuracy()
        acc_metric_val = tf.keras.metrics.BinaryAccuracy()
    else:
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        acc_metric_train = tf.keras.metrics.SparseCategoricalAccuracy()
        acc_metric_val = tf.keras.metrics.SparseCategoricalAccuracy()

    mae_metric_train = tf.keras.metrics.MeanAbsoluteError()
    mae_metric_val = tf.keras.metrics.MeanAbsoluteError()

    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss_value = loss_fn(y_batch, logits)

        grads = tape.gradient(loss_value, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        acc_metric_train.update_state(y_batch, logits)
        if is_binary:
            mae_metric_train.update_state(y_batch, logits)
        else:
            pred_classes = tf.cast(tf.argmax(logits, axis=1), tf.float32)
            y_true_f32 = tf.cast(y_batch, tf.float32)
            mae_metric_train.update_state(y_true_f32, pred_classes)

        return loss_value

    @tf.function
    def test_step(x_batch, y_batch):
        val_logits = model(x_batch, training=False)
        acc_metric_val.update_state(y_batch, val_logits)

        if is_binary:
            mae_metric_val.update_state(y_batch, val_logits)
        else:
            pred_classes = tf.cast(tf.argmax(val_logits, axis=1), tf.float32)
            y_true_f32 = tf.cast(y_batch, tf.float32)
            mae_metric_val.update_state(y_true_f32, pred_classes)

    best_val_mae = np.inf
    best_val_acc = 0.0
    best_epoch = 0
    best_weights = None
    wait = 0

    for epoch in range(epochs):
        if epoch == fine_tune_at:
            optimizer.learning_rate.assign(fine_tune_lr)
            print(f'Fine-tuning dengan learning rate {fine_tune_lr}')

        for x_batch_train, y_batch_train in train_dataset:
            train_step(x_batch_train, y_batch_train)
        for x_batch_val, y_batch_val in val_dataset:
            test_step(x_batch_val, y_batch_val)

        train_acc = float(acc_metric_train.result().numpy())
        val_acc = float(acc_metric_val.result().numpy())
        train_mae = float(mae_metric_train.result().numpy())
        val_mae = float(mae_metric_val.result().numpy())

        improved = val_mae < (best_val_mae - min_delta)
        if improved:
            best_val_mae = val_mae
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_weights = [weight.copy() for weight in model.get_weights()]
            wait = 0
        else:
            wait += 1

        if epoch == epochs - 1 or epoch % 5 == 0 or improved:
            print(
                f'Epoch {epoch + 1}: '
                f'Train Acc {train_acc:.4f}, '
                f'Val Acc {val_acc:.4f}, '
                f'Train MAE {train_mae:.4f}, '
                f'Val MAE {val_mae:.4f}'
            )

        if wait >= patience:
            print(f'Early stopping di epoch {epoch + 1}.')
            break

        acc_metric_train.reset_state()
        mae_metric_train.reset_state()
        acc_metric_val.reset_state()
        mae_metric_val.reset_state()

    if best_weights is not None:
        model.set_weights(best_weights)

    print(
        f'Best model: epoch {best_epoch}, '
        f'Val Acc {best_val_acc:.4f}, '
        f'Val MAE {best_val_mae:.4f}'
    )
    metrics = {
        'best_epoch': best_epoch,
        'val_acc': best_val_acc,
        'val_mae': best_val_mae,
    }
    return model, metrics

## 5. Inference Script

In [6]:
def inference(model, scaler, input_dict, num_classes):
    annual_income = max(input_dict['annual_income_idr'], 1)
    loan_to_income = input_dict['loan_amount_idr'] / annual_income
    expenses_to_income = input_dict['monthly_expenses_idr'] / (annual_income / 12)
    savings_to_income = input_dict['savings_balance_idr'] / annual_income
    debt_to_income = input_dict['monthly_debt_payment_idr'] / (annual_income / 12)

    raw_input = np.array([[
        loan_to_income, expenses_to_income, savings_to_income, debt_to_income,
        input_dict['credit_score'], input_dict['previous_default_count'],
        input_dict['loan_duration_months'], input_dict['interest_rate'],
        input_dict['age'], input_dict['employment_stability_years']
    ]])

    scaled_input = scaler.transform(raw_input)
    predictions = model.predict(scaled_input)[0]

    pred_class = int(np.argmax(predictions))
    prob = predictions[pred_class]

    risk_labels = {
        0: 'Low (Aman)',
        1: 'Medium (Perlu Perhatian)',
        2: 'High (Berisiko Tinggi)'
    }
    risk_label = risk_labels[pred_class]

    print(f'Probabilitas Kelas {pred_class} : {prob:.4f}')
    print(f'Profil Risiko            : {risk_label}')

    return predictions, risk_label

## 6. Execution

In [7]:
train_dataset, test_dataset, scaler, features, num_classes = load_labeled_data(LABELED_DATA_PATH)

candidate_seeds = [42, 7, 123, 2026]
best_model = None
best_metrics = None

for seed in candidate_seeds:
    print(f'\nTraining candidate seed {seed}')
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    model = create_model(len(features), num_classes=num_classes)
    candidate_model, candidate_metrics = train_model(
        model,
        train_dataset,
        test_dataset,
        num_classes=num_classes,
        epochs=60
    )

    if best_metrics is None or candidate_metrics['val_mae'] < best_metrics['val_mae']:
        best_model = candidate_model
        best_metrics = candidate_metrics

trained_model = best_model
print(
    f"\nSelected model: Val Acc {best_metrics['val_acc']:.4f}, "
    f"Val MAE {best_metrics['val_mae']:.4f}, "
    f"epoch {best_metrics['best_epoch']}"
)
trained_model.save('risk_profile_model.keras')


Training candidate seed 42


Epoch 1: Train Acc 0.7835, Val Acc 0.9320, Train MAE 0.3307, Val MAE 0.0938
Epoch 2: Train Acc 0.8920, Val Acc 0.9490, Train MAE 0.1577, Val MAE 0.0674


Epoch 3: Train Acc 0.9065, Val Acc 0.9640, Train MAE 0.1347, Val MAE 0.0459


Epoch 5: Train Acc 0.9195, Val Acc 0.9690, Train MAE 0.1160, Val MAE 0.0410
Epoch 6: Train Acc 0.9258, Val Acc 0.9720, Train MAE 0.1082, Val MAE 0.0352


Epoch 11: Train Acc 0.9362, Val Acc 0.9490, Train MAE 0.0927, Val MAE 0.0654
Epoch 12: Train Acc 0.9373, Val Acc 0.9800, Train MAE 0.0903, Val MAE 0.0273


...

Epoch 43: Train Acc 0.9560, Val Acc 0.9900, Train MAE 0.0640, Val MAE 0.0117


Epoch 46: Train Acc 0.9510, Val Acc 0.9890, Train MAE 0.0725, Val MAE 0.0127


Epoch 51: Train Acc 0.9560, Val Acc 0.9850, Train MAE 0.0640, Val MAE 0.0186


Epoch 56: Train Acc 0.9563, Val Acc 0.9850, Train MAE 0.0660, Val MAE 0.0176


Epoch 60: Train Acc 0.9603, Val Acc 0.9860, Train MAE 0.0558, Val MAE 0.0176
Best model: epoch 43, Val Acc 0.9900, Val MAE 0.0117

Selected model: Val Acc 0.9900, Val MAE 0.0117, epoch 55


## 7. Inference Example

In [8]:
sample_input = {
    'annual_income_idr': 120_000_000,
    'loan_amount_idr': 35_000_000,
    'monthly_expenses_idr': 4_500_000,
    'savings_balance_idr': 25_000_000,
    'monthly_debt_payment_idr': 2_000_000,
    'credit_score': 720,
    'previous_default_count': 0,
    'loan_duration_months': 24,
    'interest_rate': 8.5,
    'age': 32,
    'employment_stability_years': 6
}

predictions, risk_label = inference(trained_model, scaler, sample_input, num_classes)
predictions, risk_label

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step


Probabilitas Kelas 0 : 1.0000
Profil Risiko            : Low (Aman)


(array([9.9998403e-01, 1.3128221e-05, 2.8656948e-06], dtype=float32),
 'Low (Aman)')